In [11]:
print("hello")

hello


In [12]:
# %%
import yaml
from pathlib import Path

DATASET_DIR = Path("dataset")
YAML_PATH = DATASET_DIR / "data.yaml"

# 1. Load the current yaml
with open(YAML_PATH, "r") as f:
    data_cfg = yaml.safe_load(f)

print("🔧 Fixing Roboflow's relative path issue...")

# 2. Overwrite the paths to be correctly relative to the data.yaml file
data_cfg["train"] = "train/images"
data_cfg["val"] = "valid/images"
data_cfg["test"] = "test/images"

# 3. Save the corrected yaml back to the file
with open(YAML_PATH, "w") as f:
    yaml.dump(data_cfg, f, sort_keys=False)

print("✅ Fixed data.yaml! New contents:")
print(yaml.dump(data_cfg, sort_keys=False))

# 4. Verify the counts again
print("\n📊 Re-checking Dataset Counts:")
for split, folder in [("Train", "train"), ("Val", "valid"), ("Test", "test")]:
    img_dir = DATASET_DIR / folder / "images"
    if img_dir.exists():
        count = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        print(f"  {split}: {count} images ✅")
    else:
        print(f"  {split}: ❌ Folder not found at {img_dir}")

🔧 Fixing Roboflow's relative path issue...
✅ Fixed data.yaml! New contents:
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- Pepsi Bottle
roboflow:
  workspace: muhammad-saad-ali
  project: my-first-project-g8ifu
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/muhammad-saad-ali/my-first-project-g8ifu/dataset/1


📊 Re-checking Dataset Counts:
  Train: 90 images ✅
  Val: 6 images ✅
  Test: 4 images ✅


In [13]:
# %%
import cv2
import random
import albumentations as A

TARGET_TRAIN = 90
TRAIN_IMG_DIR = DATASET_DIR / "train" / "images"
TRAIN_LBL_DIR = DATASET_DIR / "train" / "labels"

def augment_dataset():
    img_paths = list(TRAIN_IMG_DIR.glob("*.jpg")) + list(TRAIN_IMG_DIR.glob("*.png"))
    current = len(img_paths)
    
    if current >= TARGET_TRAIN:
        print(f"✅ Already have {current} images. Skipping augmentation.")
        return

    needed = TARGET_TRAIN - current
    print(f"🔧 Generating {needed} augmented images to reach {TARGET_TRAIN}...")

    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.Affine(rotate=(-15, 15), scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3)
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

    count = 0
    while count < needed:
        base_img = random.choice(img_paths)
        base_lbl = TRAIN_LBL_DIR / (base_img.stem + ".txt")
        
        if not base_lbl.exists(): continue
            
        img = cv2.imread(str(base_img))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        bboxes, labels = [], []
        with open(base_lbl, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    labels.append(int(parts[0]))
                    bboxes.append([float(x) for x in parts[1:]])
                    
        if not bboxes: continue
            
        try:
            aug = transform(image=img, bboxes=bboxes, class_labels=labels)
        except Exception as e:
            continue
            
        new_name = f"aug_{count:04d}.jpg"
        new_img_path = TRAIN_IMG_DIR / new_name
        new_lbl_path = TRAIN_LBL_DIR / f"aug_{count:04d}.txt"
        
        cv2.imwrite(str(new_img_path), cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR))
        with open(new_lbl_path, "w") as f:
            for cls, box in zip(aug['class_labels'], aug['bboxes']):
                f.write(f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")
                
        count += 1
        if count % 10 == 0: print(f"  Created {count}...")
        
    print(f"✅ Augmentation complete! Total train images: {len(list(TRAIN_IMG_DIR.glob('*.jpg')))}")

augment_dataset()

✅ Already have 90 images. Skipping augmentation.


In [14]:
# %%
from ultralytics import YOLO

# Load YOLOv8 Nano (best for small datasets)
model = YOLO("yolov8n.pt")

# Train
results = model.train(
    data=str(YAML_PATH),
    epochs=150,
    imgsz=640,
    batch=16,          # Reduce to 8 if you get CUDA OOM
    patience=0,       # Early stopping
    save=True,
    project="runs/bottle_detect",
    name="exp3",
    # Complementary built-in augmentation
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
    verbose=True
)

print("✅ Training finished!")

Ultralytics 8.4.69  Python-3.13.13 torch-2.12.0+cpu CPU (Intel Core i7-8750H 2.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=0, perspectiv

In [15]:
# %%
import matplotlib.pyplot as plt
import glob

# 1. Run validation on the test split
print("📈 Running evaluation on test set...")
val_results = model.val(data=str(YAML_PATH), split="test")

# 2. Inference visualization
test_imgs = list((DATASET_DIR / "test" / "images").glob("*.jpg")) + list((DATASET_DIR / "test" / "images").glob("*.png"))

if test_imgs:
    img_path = str(test_imgs[0])
    print(f"\n🔍 Running inference on: {img_path}")
    
    preds = model(img_path, conf=0.5)
    annotated = preds[0].plot()
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("YOLOv8 Detection Result")
    plt.show()
else:
    print("⚠️ No test images found.")

📈 Running evaluation on test set...
Ultralytics 8.4.69  Python-3.13.13 torch-2.12.0+cpu CPU (Intel Core i7-8750H 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 196.459.8 MB/s, size: 152.4 KB)
val: Scanning C:\Users\Dell\Documents\NIAI\Intern tasks\bottle Detection YOLO\Dataset\test\labels.cache... 4 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 599.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.0it/s 0.5s
                   all          4          4      0.959        0.5      0.523      0.404
Speed: 1.5ms preprocess, 98.7ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to C:\Users\Dell\Documents\NIAI\Intern tasks\bottle Detection YOLO\runs\detect\val-3

🔍 Running inference on: dataset\test\images\IMG_20260616_112056_539-673903658_jpg.rf.60a5b113fc8dfdafe4c478103a48cde9.jpg

image 1/1 c:\User

<Figure size 1000x800 with 1 Axes>

In [16]:
# %%
# Load the best trained weights
best_model = YOLO("runs/detect/runs/bottle_detect/exp1/weights/best.pt")

# Export to ONNX (works in Python, C++, Web, Android, iOS)
print("📤 Exporting to ONNX...")
best_model.export(format="onnx", imgsz=640)

# Optional: Export to TensorRT (NVIDIA GPUs only)
# best_model.export(format="engine", imgsz=640)

print("✅ Export complete! Check runs/bottle_detect/exp1/weights/")

📤 Exporting to ONNX...
Ultralytics 8.4.69  Python-3.13.13 torch-2.12.0+cpu CPU (Intel Core i7-8750H 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\runs\bottle_detect\exp1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (23.3 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.8s, saved as 'runs\detect\runs\bottle_detect\exp1\weights\best.onnx' (11.7 MB)

Export complete (2.2s)
Results saved to C:\Users\Dell\Documents\NIAI\Intern tasks\bottle Detection YOLO\runs\detect\runs\bottle_detect\exp1\weights\best.onnx
Predict:         yolo predict task=detect model=runs\detect\runs\bottle_detect\exp1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs\detect\runs\bottle_detect\exp1\weights\best.onnx imgsz=640 data=dataset\data.yaml  
Visualize:       https://netron.app
✅ Export complet

In [17]:
from ultralytics import YOLO
from pathlib import Path

# ---- 1. Load your trained weights and confirm the class names ----
WEIGHTS_PATH = "runs/detect/runs/bottle_detect/exp1/weights/best.pt"  # adjust if yours saved elsewhere
model = YOLO(WEIGHTS_PATH)
print("Loaded weights from:", WEIGHTS_PATH)
print("Model classes:", model.names)
print()

# ---- 2. Re-check validation metrics ----
print("Re-running validation on test split...")
val_results = model.val(data=str(YAML_PATH), split="test")
print()

# ---- 3. Try detecting on a TRAINING image at an extremely low threshold ----
train_imgs = list((DATASET_DIR / "train" / "images").glob("*.jpg")) + \
             list((DATASET_DIR / "train" / "images").glob("*.png"))

if train_imgs:
    test_img = str(train_imgs[0])
    print(f"Testing on training image: {test_img}")
    results = model.predict(test_img, conf=0.001)
    boxes = results[0].boxes
    print(f"Boxes found at conf=0.001: {len(boxes) if boxes is not None else 0}")
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            print(f"  class={model.names[int(box.cls[0])]}, conf={float(box.conf[0]):.4f}")
else:
    print("No training images found.")

Loaded weights from: runs/detect/runs/bottle_detect/exp1/weights/best.pt
Model classes: {0: 'Pepsi Bottle'}

Re-running validation on test split...
Ultralytics 8.4.69  Python-3.13.13 torch-2.12.0+cpu CPU (Intel Core i7-8750H 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 387.9119.4 MB/s, size: 152.4 KB)
val: Scanning C:\Users\Dell\Documents\NIAI\Intern tasks\bottle Detection YOLO\Dataset\test\labels.cache... 4 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 762.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.0it/s 0.5s
                   all          4          4    0.00167        0.5       0.37      0.177
Speed: 2.4ms preprocess, 94.3ms inference, 0.0ms loss, 9.4ms postprocess per image
Results saved to C:\Users\Dell\Documents\NIAI\Intern tasks\bottle Detection YOLO\runs\detect\val-4

Testing on training image: 